# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id` fields as required.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

* https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load Croissant dataset
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

# Show overview
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print(f"Version: {getattr(metadata, 'version', '')}")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"License: {getattr(metadata, 'license', '')}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s. 

We'll explore the Croissant metadata to summarize available schema elements, referencing all by `@id`.

In [ ]:
# List available RecordSets and their `@id`
record_set_ids = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        rs_id = getattr(rs, '@id', getattr(rs, 'id', None))
        rs_type = getattr(rs, '@type', getattr(rs, 'type', None))
        print(f"RecordSet: @id={rs_id}, @type={rs_type}")
        record_set_ids.append(rs_id)
else:
    print("No RecordSets found in metadata. Dataset may contain only one main RecordSet.")

# If recordSet is empty, try to load default RecordSets from the underlying schema by inspecting the ds.schema
if not record_set_ids:
    from mlcroissant.utils.jsonld import expand_jsonld
    expanded = expand_jsonld(ds._jsonld)
    record_sets = [obj for obj in expanded if obj.get('@type') == 'RecordSet']
    for rs in record_sets:
        rs_id = rs.get('@id')
        print(f"RecordSet: @id={rs_id}, @type={rs.get('@type')}")
        record_set_ids.append(rs_id)

# List fields and columns for each RecordSet
# Use @id references throughout
field_ids = {}
for rs_id in record_set_ids:
    # Inspect schema for fields
    rs_schema = next((obj for obj in expanded if obj.get('@id') == rs_id), None)
    if rs_schema and 'field' in rs_schema:
        fields = rs_schema['field'] if isinstance(rs_schema['field'], list) else [rs_schema['field']]
        field_ids[rs_id] = []
        print(f"Fields for RecordSet @id={rs_id}:")
        for f in fields:
            if isinstance(f, dict):
                f_id = f.get('@id', None)
                f_name = f.get('name', '')
                print(f"    Field: @id={f_id}, name={f_name}")
                field_ids[rs_id].append(f_id)
            else:
                print(f"    Field @id={f}")
                field_ids[rs_id].append(f)
    else:
        print(f"No fields found for RecordSet @id={rs_id}.")

## 3. Data Extraction
Load data from each RecordSet into DataFrames for analysis. Use RecordSet and Field `@id`s from the overview.

In [ ]:
# Extract data from all record sets discovered
import pprint
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading data for RecordSet @id={record_set_id}...")
    records = list(ds.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"No records found for @id={record_set_id}.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for RecordSet @id={record_set_id}: {df.columns.tolist()}")
    print(f"First 5 rows for RecordSet @id={record_set_id}:")
    pprint.pprint(df.head().to_dict())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing, and grouping data using Field `@id`s for reference.

Let's pick one RecordSet and its fields for demonstration. We'll look for numeric fields (e.g., age) and group by categorical (e.g., sex).

In [ ]:
# Example: Use the first available RecordSet
if dataframes:
    selected_record_set = list(dataframes.keys())[0]
    df = dataframes[selected_record_set]
    print(f"Working with RecordSet @id={selected_record_set}")

    # Search for likely numeric fields by scanning column names
    import re
    numeric_fields = [col for col in df.columns if re.search(r'age|years|interval|score|number|count', col, re.I) or pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        numeric_field_id = df.select_dtypes('number').columns.tolist()[0] if len(df.select_dtypes('number').columns) > 0 else df.columns[0]
    print(f"Numeric field selected for analysis: {numeric_field_id}")

    # Filter on a threshold (example threshold=10)
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Search for possible group field (e.g., sex, anatomical_location, comorbidity)
        group_fields = [col for col in df.columns if re.search(r'sex|gender|location|site|comorbidity', col, re.I) or pd.api.types.is_object_dtype(df[col])]
        group_field_id = group_fields[0] if group_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"Numeric field {numeric_field_id} not found in DataFrame columns.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize numeric distributions and relationships using pandas and matplotlib.

We'll plot the histogram of the selected numeric field and a boxplot grouped by the selected categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered > {threshold})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated step-by-step data loading, overview, extraction, processing, and visualization for the FAIR² colorectal cancer survivor dataset using the `mlcroissant` library. 

All entities—record sets, fields, columns—were referenced by their `@id` per schema standards. The exploratory analysis suggests the dataset is suitable for clinical stratification research and modeling, though the limited sample size and single-center data are important caveats for interpretation.

Further analysis could refine groupings, validate findings, or extend to predictive modeling tasks.